# Bronze schema creation

Creates or verifies the source-qualified Bronze Delta tables declared by the landing manifest, plus the ingestion audit and quarantine control tables. This notebook never loads source rows.

Run this notebook before `EDU_BR_Ingestion`. Schema drift is rejected instead of being merged implicitly.

In [ ]:
# Deployment parameters
source_root = "Files/landing/education-analytics"
manifest_path = f"{source_root}/manifest.json"

import json
import re

from pyspark.sql import types as T

## Physical contract

Source columns remain nullable strings in Bronze. Lineage columns have stable Spark types. Existing tables are accepted only when their ordered column names and data types match the generated contract.

In [ ]:
def read_json(path: str) -> dict:
    return json.loads(notebookutils.fs.head(path, 1024 * 1024))


def normalized_table_name(source: str, relative_path: str) -> str:
    stem = relative_path.rsplit("/", 1)[-1].rsplit(".", 1)[0]
    return re.sub(r"[^a-z0-9]+", "_", f"{source}_{stem}".lower()).strip("_")


def source_table_schema(columns: list[str]) -> T.StructType:
    return T.StructType(
        [T.StructField(column, T.StringType(), True) for column in columns]
        + [
            T.StructField("LoadBatchId", T.StringType(), True),
            T.StructField("SourceSystemName", T.StringType(), True),
            T.StructField("SourceEntityName", T.StringType(), True),
            T.StructField("SourceInstanceId", T.StringType(), True),
            T.StructField("SourceFileName", T.StringType(), True),
            T.StructField("IngestedAtUtc", T.TimestampType(), True),
            T.StructField("ObservationCutoff", T.TimestampType(), True),
            T.StructField("RawRecordHash", T.StringType(), True),
        ]
    )


audit_schema = T.StructType([
    T.StructField("LoadBatchId", T.StringType(), False),
    T.StructField("SourceSystemName", T.StringType(), False),
    T.StructField("SourceEntityName", T.StringType(), False),
    T.StructField("SourceInstanceId", T.StringType(), False),
    T.StructField("SourceFileName", T.StringType(), False),
    T.StructField("TargetTable", T.StringType(), False),
    T.StructField("ExpectedRows", T.LongType(), False),
    T.StructField("ActualRows", T.LongType(), False),
    T.StructField("AcceptedRows", T.LongType(), False),
    T.StructField("RejectedRows", T.LongType(), False),
    T.StructField("ExpectedSha256", T.StringType(), False),
    T.StructField("ActualSha256", T.StringType(), False),
    T.StructField("Status", T.StringType(), False),
    T.StructField("StartedAtUtc", T.TimestampType(), False),
    T.StructField("FinishedAtUtc", T.TimestampType(), False),
    T.StructField("ErrorMessage", T.StringType(), True),
])

quarantine_schema = T.StructType([
    T.StructField("LoadBatchId", T.StringType(), True),
    T.StructField("SourceSystemName", T.StringType(), True),
    T.StructField("SourceEntityName", T.StringType(), True),
    T.StructField("SourceInstanceId", T.StringType(), True),
    T.StructField("SourceFileName", T.StringType(), True),
    T.StructField("IngestedAtUtc", T.TimestampType(), True),
    T.StructField("RawRecordHash", T.StringType(), True),
    T.StructField("ValidationErrors", T.ArrayType(T.StringType()), True),
    T.StructField("RawPayload", T.StringType(), True),
    T.StructField("CorruptRecord", T.StringType(), True),
])


def schema_signature(schema: T.StructType) -> list[tuple[str, str]]:
    return [(field.name, field.dataType.simpleString()) for field in schema.fields]


def ensure_table(table_name: str, expected_schema: T.StructType) -> str:
    if not spark.catalog.tableExists(table_name):
        (spark.createDataFrame([], expected_schema).write
         .format("delta").mode("errorifexists").saveAsTable(table_name))
        return "CREATED"
    actual_schema = spark.table(table_name).schema
    if schema_signature(actual_schema) != schema_signature(expected_schema):
        raise ValueError(
            f"Unsafe Bronze schema drift for {table_name}: "
            f"expected {schema_signature(expected_schema)}, received {schema_signature(actual_schema)}"
        )
    return "VERIFIED"

## Create or verify Bronze tables

The manifest is the source of truth for source-table names and columns. Duplicate paths, duplicate normalized table names, unexpected source systems, or an unexpected table count stop deployment.

In [ ]:
manifest = read_json(manifest_path)
tables = manifest.get("tables", [])
allowed_sources = {"PowerSchool", "Canvas", "Moodle", "NWEA"}

if len(tables) != 28:
    raise ValueError(f"Manifest declares {len(tables)} tables; expected 28")
if {table["source"] for table in tables} != allowed_sources:
    raise ValueError("Manifest source systems do not match the Bronze contract")
paths = [table["path"] for table in tables]
if len(paths) != len(set(paths)):
    raise ValueError("Manifest contains duplicate table paths")

table_names = [normalized_table_name(table["source"], table["path"]) for table in tables]
if len(table_names) != len(set(table_names)):
    raise ValueError("Manifest paths produce duplicate Bronze table names")

results = []
for table, table_name in zip(tables, table_names):
    state = ensure_table(table_name, source_table_schema(table["columns"]))
    results.append({"Table": table_name, "Kind": "Source", "State": state})

results.append({"Table": "ingestion_audit", "Kind": "Control", "State": ensure_table("ingestion_audit", audit_schema)})
results.append({"Table": "ingestion_quarantine", "Kind": "Control", "State": ensure_table("ingestion_quarantine", quarantine_schema)})

display(spark.createDataFrame(results).orderBy("Kind", "Table"))
assert len(results) == 30, f"Prepared {len(results)} of 30 Bronze tables"
print("Bronze schema is ready: 28 source tables and 2 control tables")